###Setup

In [ ]:
# Installazione delle librerie
!pip install -q chromadb sentence-transformers langchain langchain-community langchain-text-splitters langchain-google-genai
!pip install -q rank-bm25            # per la ricerca lessicale (hybrid search)
!pip install -q tiktoken             # per il chunking a token
!pip install -q ragas datasets       # solo per la sezione di valutazione

print("Librerie installate.")

In [ ]:
import os
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if GEMINI_API_KEY:
    print("Chiave Gemini presente.")
else:
    print("Nessuna chiave Gemini: imposta la variabile d'ambiente GEMINI_API_KEY.")

##1. Ingestion

In [ ]:
#Creazione documenti di esempio con metadati (origine, data, data di validità, tipo)

documenti = [
    {
        "id": "POL-001",
        "testo": """PROCEDURA DI RICHIESTA ACCESSO AI DATI DEI CLIENTI (v3.2)
La richiesta di accesso ai dati dei clienti deve essere inoltrata tramite il portale interno DataRequest.
Ogni richiesta richiede: (1) giustificazione del business need, (2) approvazione del data owner,
(3) verifica del consenso del cliente nel registro consensi. L'accesso viene concesso per un periodo
massimo di 90 giorni, rinnovabile. Gli accessi sono tracciati e soggetti ad audit trimestrale.""",
        "origine": "Portale Compliance interno",
        "data": "2025-11-10",
        "data_validita": "2026-11-10",
        "tipo": "policy",
    },
    {
        "id": "GUI-002",
        "testo": """GUIDA OPERATIVA — GESTIONE CONSENSI GDPR
Prima di accedere ai dati di un cliente, verificare sempre lo stato del consenso nel registro.
Il GDPR (Reg. UE 2016/679) richiede base giuridica valida per ogni trattamento. Per l'accesso interno
a fini di analisi, la base è tipicamente il legittimo interesse, che va bilanciato con i diritti
dell'interessato. Documentare sempre la finalità. Conservare i dati non oltre il necessario.""",
        "origine": "Manuale DPO",
        "data": "2025-09-01",
        "data_validita": "2026-09-01",
        "tipo": "guida",
    },
    {
        "id": "REG-003",
        "testo": """REGISTRO CONSENSI — ISTRUZIONI DI CONSULTAZIONE
Il registro consensi centralizza lo stato di consenso di ogni cliente per ciascuna finalità
(marketing, profilazione, analisi interna). Si consulta tramite l'ID cliente. Uno stato 'revocato'
blocca qualsiasi accesso ai dati per quella finalità. Il registro è aggiornato in tempo reale e
costituisce la fonte autoritativa in caso di audit o richiesta dell'autorità Garante.""",
        "origine": "Sistema Registro Consensi",
        "data": "2025-12-01",
        "data_validita": "2026-06-01",
        "tipo": "registro",
    },
    {
        "id": "POL-OLD-000",
        "testo": """PROCEDURA ACCESSO DATI CLIENTI (v1.0 - OBSOLETA)
La richiesta va inviata via email al responsabile IT. Nessuna scadenza automatica sull'accesso.""",
        "origine": "Archivio storico",
        "data": "2021-03-15",
        "data_validita": "2023-03-15",   # scaduta: proveremo a testare il filtro sulle date
        "tipo": "policy",
    },
]

print(f"Caricati {len(documenti)} documenti.")
for d in documenti:
    print(f"  {d['id']} | {d['tipo']} | valido fino al {d['data_validita']}")

## 2. Indicizzazione

In [ ]:
#CHUNKING a Token (invece che a caratteri) per somiglianza con utilizzo reale con LLM

from langchain_text_splitters import TokenTextSplitter
from langchain_core.documents import Document

splitter_token = TokenTextSplitter(chunk_size=55, chunk_overlap=20)


langchain_docs = [
    Document(
        page_content=doc_data['testo'],
        metadata={key: value for key, value in doc_data.items() if key != 'testo'}
    )
    for doc_data in documenti
]

print(f"Creati {len(langchain_docs)} oggetti Document di LangChain.")

# Divido i documenti usando lo splitter e conservo i metadati
chunks_with_metadata = splitter_token.split_documents(langchain_docs)

print(f"Numero totale di chunk con metadati: {len(chunks_with_metadata)}")

In [ ]:
from collections import Counter
conteggio = Counter(c.metadata['id'] for c in chunks_with_metadata)
print(conteggio)

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
#disattiviamo l'import GoogleGenerativeAIEmbeddings per far girare il chunking gratuitamente in locale
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

# Modello di embedding LOCALE (gratuito su Colab).
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Client Google Generative AI per gli embedding (alternativa all'embedder locale)
# Assicurati che os.environ["GEMINI_API_KEY"] sia impostato correttamente.
# gemini_api_key = os.environ.get("GEMINI_API_KEY")
# if not gemini_api_key:
#     raise ValueError("La chiave API di Gemini non è impostata")

# # Cambiamo il modello di embedding a 'embedding-001' che è più comunemente supportato
# embedder_gemini = GoogleGenerativeAIEmbeddings(google_api_key=gemini_api_key, model="embedding-001")

# Client Chroma in-memory (per la produzione: chromadb.PersistentClient(path=...))
client = chromadb.Client()
collection = client.get_or_create_collection(name="documenti_aziendali")

# I chunk sono oggetti Document, si può accedere a page_content e metadata
testi = [c.page_content for c in chunks_with_metadata]

# Usa l'embedder LOCALE per generare gli embedding
embeddings = embedder.encode(testi).tolist()

# Questo è l'embedder di Gemini per generare gli embedding
# embeddings = embedder_gemini.embed_documents(testi)

# Genero ID unici per ogni chunk e uso il dizionario metadata
collection.add(
    ids=[f"{c.metadata['id']}_{i}" for i, c in enumerate(chunks_with_metadata)],
    documents=testi,
    embeddings=embeddings,
    metadatas=[c.metadata for c in chunks_with_metadata],
)

print(f"Indicizzati {collection.count()} chunk nel vector store Chroma.")

## 3. Retrieval ibrido

In [ ]:
from datetime import date

def retrieval_semantico(query, k=4, solo_validi=True):
    """Ricerca semantica con filtro opzionale sulla data di validita'."""
    query_emb = embedder.encode([query]).tolist()

    # Filtro sui metadati: Chroma supporta il parametro where.
    risultati = collection.query(query_embeddings=query_emb, n_results=k*2)

    # Stampa i risultati grezzi di Chroma (spostati qui per risolvere il NameError precedente)
    print("--- Risultati RAW da ChromaDB ---")
    print("IDs:", risultati["ids"][0])
    print("Distances:", risultati["distances"][0])
    print("---------------------------------")

    candidati = []
    conteggio_per_doc = {}

    oggi = str(date.today())

    for i in range(len(risultati["ids"][0])):
        meta = risultati["metadatas"][0][i]
        if solo_validi and meta["data_validita"] < oggi:
            continue   # scarta documenti con validita' scaduta
        valore_meta = conteggio_per_doc.get(meta["id"], 0)
        if valore_meta >= 2:
            continue
        candidati.append({
            "chunk_id": risultati["ids"][0][i],
            "testo": risultati["documents"][0][i],
            "distanza": risultati["distances"][0][i],
            **meta,
        })

        conteggio_per_doc [meta["id"]] = valore_meta + 1
    return candidati[:k]
# Test sul caso d'uso
query = "Qual e' la procedura aggiornata per la richiesta di accesso ai dati dei clienti e quali normative dobbiamo rispettare?"
candidati = retrieval_semantico(query, k=4)

print(f"Recuperati {len(candidati)} chunk (validi):\n")
for c in candidati:
    print(f"  {c['chunk_id']} | {c['tipo']} | valido fino {c['data_validita']} | dist={c['distanza']:.3f}")
    print(f"     {c['testo'][:70]}...")

## 4. Pipeline di generazione

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
modello = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

In [ ]:
input_ids = tokenizer("Translate to Italian: The cat is on the table.", return_tensors="pt").input_ids
output = modello.generate(input_ids, max_new_tokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
def costruisci_prompt(query, candidati):
    testi = " ".join(c["testo"] for c in candidati)
    prompt = f"Contesto: {testi}\n\nDomanda: {query}\n\nRisposta:"
    return prompt

In [ ]:
prompt_test = costruisci_prompt(query, candidati)
print(prompt_test)

In [ ]:
input_ids = tokenizer(prompt_test, return_tensors="pt").input_ids
output = modello.generate(input_ids, max_new_tokens=150, num_beams=4)
print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
def genera_risposta(query, candidati):
    if not candidati:
        return {"risposta": "Non ci sono fonti", "fonti": [], "confidenza": 0.0}  # gestisci il caso "nessuna fonte" come vuoi tu

    prompt = costruisci_prompt(query, candidati)

    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    output = modello.generate(input_ids, max_new_tokens=150, num_beams=4)
    risposta_testo = tokenizer.decode(output[0], skip_special_tokens=True)

    # qui calcoli la confidenza dalla distanza media dei candidati (stessa idea di prima:
    # distanza bassa = confidenza alta)

    # qui costruisci la lista fonti da candidati (id, tipo, data_validita)

    return {"risposta": ..., "fonti": ..., "confidenza": ...}

In [ ]:
confidence = [c["distanza"] for c in candidati]

sum_candidati = sum(confidence)
len_candidati = len(confidence)
mean_candidati = sum(confidence) / len_candidati

print(sum_candidati)
print(len_candidati)
print(mean_candidati)

confidenza = 1 - (mean_candidati / 2)
print(confidenza)

In [ ]:
fonti = [{"id": c["id"], "tipo":c["tipo"], "data_validita":c["data_validita"]} for c in candidati]
print(fonti)

In [ ]:
def genera_risposta(query, candidati):
    if not candidati:
        return {"risposta": "Non ci sono fonti", "fonti": [], "confidenza": 0.0}

    prompt = costruisci_prompt(query, candidati)

    input_ids = tokenizer(prompt, return_tensors="pt").input_ids
    output = modello.generate(input_ids, max_new_tokens=150, num_beams=4)
    risposta_testo = tokenizer.decode(output[0], skip_special_tokens=True)

    confidence = [c["distanza"] for c in candidati]
    mean_candidati = sum(confidence) / len(confidence)
    confidenza = 1 - (mean_candidati / 2)

    fonti = [{"id": c["id"], "tipo": c["tipo"], "data_validita": c["data_validita"]} for c in candidati]

    return {"risposta": risposta_testo, "fonti": fonti, "confidenza": round(confidenza, 2)}

In [ ]:
risultato = genera_risposta(query, candidati)
print("RISPOSTA:", risultato["risposta"])
print("FONTI:", risultato["fonti"])
print("CONFIDENZA:", risultato["confidenza"])